# Battery Dataset Analysis and Unified CSV Builder

This notebook analyzes the datasets under `data/` and creates `data/unified.csv`.

Supported sources:
- NASA PCoE battery ageing (`.mat`)
- CALCE battery cycling files (`.xlsx`, `.xls`, `.txt`, `.csv`)
- RWTH Mobile2Mobile selected smartphone telemetry (`.parquet`)

## Install dependencies

In [46]:
%pip install -q numpy pandas scipy openpyxl xlrd pyarrow h5py tqdm

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and locate

In [47]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import json
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.io import loadmat
from IPython.display import display

CWD = Path.cwd()
candidates = [CWD, CWD / "data", CWD.parent / "data"]

DATA_DIR = next(
    (
        p.resolve()
        for p in candidates
        if (p / "nasa").exists()
        and (p / "calce").exists()
        and (p / "mobile2mobile").exists()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        f"Cannot locate data/ with nasa/, calce/, mobile2mobile/. CWD={CWD}"
    )

NASA_DIR = DATA_DIR / "nasa"
CALCE_DIR = DATA_DIR / "calce"
MOBILE_SELECTED_DIR = DATA_DIR / "mobile2mobile" / "selected_data"
OUTPUT_PARQUET = DATA_DIR / "unified.parquet"

print("Project data directory:", DATA_DIR)
print("NASA:", NASA_DIR.exists())
print("CALCE:", CALCE_DIR.exists())
print("Mobile2Mobile selected data:", MOBILE_SELECTED_DIR.exists())
print("Output:", OUTPUT_PARQUET)

Project data directory: /home/raj/Documents/Projects/PBL/data
NASA: True
CALCE: True
Mobile2Mobile selected data: True
Output: /home/raj/Documents/Projects/PBL/data/unified.parquet


## 2. Inspect folder structure

In [48]:
def show_tree(root, max_depth=3, max_files_per_dir=25):
    print(f"\n{root}/")
    if not root.exists():
        print("  [MISSING]")
        return
    dirs = defaultdict(list)
    for p in root.rglob("*"):
        rel = p.relative_to(root)
        if len(rel.parts) <= max_depth:
            dirs[rel.parent].append(p)
    for parent in sorted(dirs, key=lambda x: (len(x.parts), str(x))):
        items = sorted(dirs[parent], key=lambda x: (not x.is_dir(), x.name))
        for p in items[:max_files_per_dir]:
            depth = len(p.relative_to(root).parts)
            print("  " * depth + p.name + ("/" if p.is_dir() else ""))
        if len(items) > max_files_per_dir:
            print("  " * (len(parent.parts)+1) + f"... {len(items)-max_files_per_dir} more")

show_tree(NASA_DIR, max_depth=3)
show_tree(CALCE_DIR, max_depth=3)
show_tree(MOBILE_SELECTED_DIR, max_depth=2)


/home/raj/Documents/Projects/PBL/data/nasa/
  extracted/
    1. BatteryAgingARC-FY08Q4/
    2. BatteryAgingARC_25_26_27_28_P1/
    3. BatteryAgingARC_25-44/
    4. BatteryAgingARC_45_46_47_48/
    5. BatteryAgingARC_49_50_51_52/
    6. BatteryAgingARC_53_54_55_56/
      B0005.mat
      B0006.mat
      B0007.mat
      B0018.mat
      README.txt
      B0025.mat
      B0026.mat
      B0027.mat
      B0028.mat
      README.txt
      B0025.mat
      B0026.mat
      B0027.mat
      B0028.mat
      B0029.mat
      B0030.mat
      B0031.mat
      B0032.mat
      B0033.mat
      B0034.mat
      B0036.mat
      B0038.mat
      B0039.mat
      B0040.mat
      B0041.mat
      B0042.mat
      B0043.mat
      B0044.mat
      README_25_26_27_28.txt
      README_29_30_31_32.txt
      README_33_34_36.txt
      README_38_39_40.txt
      README_41_42_43_44.txt
      B0045.mat
      B0046.mat
      B0047.mat
      B0048.mat
      README_45_46_47_48.txt
      B0049.mat
      B0050.mat
      B0051.mat
    

## 3. File inventory and storage summary

In [49]:
SOURCE_ROOTS = {
    "NASA": NASA_DIR,
    "CALCE": CALCE_DIR,
    "MOBILE2MOBILE": MOBILE_SELECTED_DIR,
}
records = []
for source, root in SOURCE_ROOTS.items():
    for p in root.rglob("*"):
        if p.is_file():
            records.append({
                "source": source,
                "path": str(p.relative_to(DATA_DIR)),
                "extension": p.suffix.lower() or "[none]",
                "size_MB": p.stat().st_size / 1024**2,
            })
file_inventory = pd.DataFrame(records)
print("Files found:", len(file_inventory))
display(file_inventory.groupby(["source", "extension"]).agg(
    files=("path", "count"), size_MB=("size_MB", "sum")
).round(2))
print("\nLargest 25 files:")
display(file_inventory.nlargest(25, "size_MB"))

Files found: 164


files  size_MB
source        extension                
CALCE         .txt           2     2.19
              .xlsx         86   178.75
MOBILE2MOBILE .parquet      28   193.08
NASA          .mat          38   199.93
              .txt          10     0.02


Largest 25 files:


,source,path,extension,size_MB
7,NASA,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0007...,.mat,15.305958
5,NASA,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0006...,.mat,15.285644
9,NASA,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,.mat,15.217661
139,MOBILE2MOBILE,mobile2mobile/selected_data/390119b7903bf350b1...,.parquet,13.669004
150,MOBILE2MOBILE,mobile2mobile/selected_data/94be8152e70c10e3f5...,.parquet,13.648797
152,MOBILE2MOBILE,mobile2mobile/selected_data/8db535f0e09cb8814f...,.parquet,13.526697
155,MOBILE2MOBILE,mobile2mobile/selected_data/204a90fb0bb4ea8b88...,.parquet,13.366133
160,MOBILE2MOBILE,mobile2mobile/selected_data/edc9dd9f4406fcff37...,.parquet,13.146398
138,MOBILE2MOBILE,mobile2mobile/selected_data/5743a4e5a5322b1691...,.parquet,12.984632
158,MOBILE2MOBILE,mobile2mobile/selected_data/3b0708ec03aea8a59e...,.parquet,12.879576


## 4. NASA MAT-file discovery and schema inspection

In [50]:
nasa_files = sorted(NASA_DIR.rglob("*.mat"))
print("NASA MAT files:", len(nasa_files))
for p in nasa_files:
    print(f"{p.relative_to(DATA_DIR)} | {p.stat().st_size/1024**2:.2f} MB")
if not nasa_files:
    raise FileNotFoundError("No .mat files found under nasa/")


def load_mat_file(path):
    try:
        return loadmat(path, simplify_cells=True)
    except (NotImplementedError, ValueError):
        try:
            import mat73
            return mat73.loadmat(str(path))
        except ImportError as exc:
            raise RuntimeError(
                f"{path.name} may be MATLAB v7.3. Install fallback: %pip install mat73"
            ) from exc


def plain(value):
    if isinstance(value, dict):
        return {k: plain(v) for k, v in value.items()}
    if hasattr(value, "_fieldnames"):
        return {k: plain(getattr(value, k)) for k in value._fieldnames}
    if isinstance(value, np.ndarray):
        if value.dtype == object:
            vals = [plain(v) for v in value.ravel()]
            return vals[0] if len(vals) == 1 else vals
        if value.size == 1:
            return value.item()
        return np.asarray(value).squeeze()
    if isinstance(value, np.generic):
        return value.item()
    return value


def get_key(mapping, *names):
    if not isinstance(mapping, dict):
        return None
    lookup = {str(k).strip().lower(): k for k in mapping}
    for name in names:
        if name.lower() in lookup:
            return mapping[lookup[name.lower()]]
    return None


def to_list(value):
    value = plain(value)
    if value is None:
        return []
    return value if isinstance(value, list) else [value]

sample_path = nasa_files[0]
mat = load_mat_file(sample_path)
keys = [k for k in mat if not k.startswith("__")]
battery = plain(mat.get(sample_path.stem))
if not isinstance(battery, dict):
    battery = next((plain(mat[k]) for k in keys if isinstance(plain(mat[k]), dict)), None)
print("\nSample:", sample_path.name)
print("MAT top-level keys:", keys)
print("Battery fields:", list(battery) if isinstance(battery, dict) else type(battery))
cycles = to_list(get_key(battery, "cycle"))
print("Number of cycles:", len(cycles))
if cycles:
    first_cycle = cycles[0]
    print("First cycle fields:", list(first_cycle) if isinstance(first_cycle, dict) else type(first_cycle))
    d = plain(get_key(first_cycle, "data"))
    print("First cycle type:", get_key(first_cycle, "type"))
    print("Cycle data fields and shapes:")
    for k, v in (d.items() if isinstance(d, dict) else []):
        try: shape = np.asarray(v).shape
        except Exception: shape = "?"
        print(f"  {k:28} {shape}")

NASA MAT files: 38
nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005.mat | 15.22 MB
nasa/extracted/1. BatteryAgingARC-FY08Q4/B0006.mat | 15.29 MB
nasa/extracted/1. BatteryAgingARC-FY08Q4/B0007.mat | 15.31 MB
nasa/extracted/1. BatteryAgingARC-FY08Q4/B0018.mat | 8.11 MB
nasa/extracted/2. BatteryAgingARC_25_26_27_28_P1/B0025.mat | 2.63 MB
nasa/extracted/2. BatteryAgingARC_25_26_27_28_P1/B0026.mat | 2.64 MB
nasa/extracted/2. BatteryAgingARC_25_26_27_28_P1/B0027.mat | 2.65 MB
nasa/extracted/2. BatteryAgingARC_25_26_27_28_P1/B0028.mat | 2.69 MB
nasa/extracted/3. BatteryAgingARC_25-44/B0025.mat | 2.63 MB
nasa/extracted/3. BatteryAgingARC_25-44/B0026.mat | 2.64 MB
nasa/extracted/3. BatteryAgingARC_25-44/B0027.mat | 2.65 MB
nasa/extracted/3. BatteryAgingARC_25-44/B0028.mat | 2.69 MB
nasa/extracted/3. BatteryAgingARC_25-44/B0029.mat | 2.75 MB
nasa/extracted/3. BatteryAgingARC_25-44/B0030.mat | 2.75 MB
nasa/extracted/3. BatteryAgingARC_25-44/B0031.mat | 2.75 MB
nasa/extracted/3. BatteryAgingARC_25-4

## 5. CALCE workbook/text inspection

In [51]:
calce_files = sorted(p for p in CALCE_DIR.rglob("*") if p.is_file()
                     and p.suffix.lower() in {".xlsx", ".xls", ".txt", ".csv"})
calce_inventory = pd.DataFrame([{
    "cell_id": p.relative_to(CALCE_DIR).parts[0],
    "filename": p.name,
    "extension": p.suffix.lower(),
    "size_MB": p.stat().st_size/1024**2,
    "path": str(p.relative_to(DATA_DIR)),
} for p in calce_files])
print("CALCE files:", len(calce_inventory))
display(calce_inventory.groupby(["cell_id", "extension"]).size().unstack(fill_value=0))

# Inspect one workbook per cell, including sheet names, column labels and a few records.
for cell in sorted(CALCE_DIR.iterdir()):
    if not cell.is_dir():
        continue
    workbooks = sorted(p for p in cell.rglob("*") if p.suffix.lower() in {".xlsx", ".xls"})
    if not workbooks:
        continue
    path = workbooks[0]
    print(f"\n--- {path.relative_to(DATA_DIR)} ---")
    try:
        xls = pd.ExcelFile(path)
        print("Sheets:", xls.sheet_names[:15])
        for sheet in xls.sheet_names[:2]:
            preview = pd.read_excel(path, sheet_name=sheet, nrows=8)
            print("Sheet:", sheet)
            print("Columns:", preview.columns.tolist())
            display(preview.head(5))
    except Exception as exc:
        print("Inspection error:", repr(exc))

CALCE files: 88


extension,.txt,.xlsx
cell_id,,
CS2_33,0,7
CS2_34,0,7
CS2_35,0,7
CS2_36,0,7
CS2_37,0,7
CS2_38,0,7
CX2_16,2,5
CX2_33,0,7
CX2_34,0,7



--- calce/CS2_33/CS2_33/CS2_33_10_26_10.xlsx ---
Sheets: ['Info', 'Channel_1-006']
Sheet: Info
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'TEST REPORT', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,TEST REPORT,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,NaN,NaN,NaN,10_15_10,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,2010-10-26 00:00:00,NaN,NaN,NaN,Mapped_Aux_Number,NaN,NaN,...,NaN,NaN,NaN,Cell Grading,NaN,NaN,Concentration,Digital I/O,NaN,NaN
2,Channel,Start_DateTime,Schedule_File_Name,Creator,Comments,Chan_Num,Item_ID,Volt,Temp,Pres,...,Aux,SmartBty,Event,Grade,Internal_Resistance(Ohm),Discharge_Capacity(Ah),Concentration,Digital Input,Digital Output,Related Humidity
3,1-006,2010-10-15 17:12:26,CS2_point5C.sdu,NaN,NaN,16,NaN,0,0,0,...,False,False,False,NaN,NaN,NaN,0,0,0,NaN


Sheet: Channel_1-006
Columns: ['Data_Point', 'Test_Time(s)', 'Date_Time', 'Step_Time(s)', 'Step_Index', 'Cycle_Index', 'Current(A)', 'Voltage(V)', 'Charge_Capacity(Ah)', 'Discharge_Capacity(Ah)', 'Charge_Energy(Wh)', 'Discharge_Energy(Wh)', 'dV/dt(V/s)', 'Internal_Resistance(Ohm)', 'Is_FC_Data', 'AC_Impedance(Ohm)', 'ACI_Phase_Angle(Deg)']


,Data_Point,Test_Time(s),Date_Time,Step_Time(s),Step_Index,Cycle_Index,Current(A),Voltage(V),Charge_Capacity(Ah),Discharge_Capacity(Ah),Charge_Energy(Wh),Discharge_Energy(Wh),dV/dt(V/s),Internal_Resistance(Ohm),Is_FC_Data,AC_Impedance(Ohm),ACI_Phase_Angle(Deg)
0,1,30.015247,2010-10-15 17:12:56,30.015248,1,1,0.000000,3.520348,0.000000,0,0.000000,0,0.000000,0,0,0,0
1,2,60.030234,2010-10-15 17:13:26,60.030235,1,1,0.000000,3.520348,0.000000,0,0.000000,0,0.000000,0,0,0,0
2,3,90.045398,2010-10-15 17:13:56,90.045398,1,1,0.000000,3.520348,0.000000,0,0.000000,0,0.000000,0,0,0,0
3,4,120.013513,2010-10-15 17:14:26,120.013514,1,1,0.000000,3.520510,0.000000,0,0.000000,0,0.000000,0,0,0,0
4,5,150.029139,2010-10-15 17:14:56,30.015209,2,1,0.550205,3.660633,0.004586,0,0.016679,0,0.001102,0,0,0,0


... (44 outputs left out)


--- calce/CX2_38/CX2_38/CX2_38_11_01_10.xlsx ---
Sheets: ['Info', 'Channel_1-005', 'Statistics_1-005']
Sheet: Info
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'TEST REPORT', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,TEST REPORT,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,NaN,NaN,NaN,10_22_10,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,2010-11-01 00:00:00,NaN,NaN,NaN,Mapped_Aux_Number,NaN,NaN,...,NaN,NaN,NaN,Cell Grading,NaN,NaN,Concentration,Digital I/O,NaN,NaN
2,Channel,Start_DateTime,Schedule_File_Name,Creator,Comments,Chan_Num,Item_ID,Volt,Temp,Pres,...,Aux,SmartBty,Event,Grade,Internal_Resistance(Ohm),Discharge_Capacity(Ah),Concentration,Digital Input,Digital Output,Related Humidity
3,1-005,2010-10-22 13:23:25,CX2_1C.sdu,NaN,NaN,16,NaN,0,0,0,...,False,False,False,NaN,NaN,NaN,0,0,0,NaN


Sheet: Channel_1-005
Columns: ['Data_Point', 'Test_Time(s)', 'Date_Time', 'Step_Time(s)', 'Step_Index', 'Cycle_Index', 'Current(A)', 'Voltage(V)', 'Charge_Capacity(Ah)', 'Discharge_Capacity(Ah)', 'Charge_Energy(Wh)', 'Discharge_Energy(Wh)', 'dV/dt(V/s)', 'Internal_Resistance(Ohm)', 'Is_FC_Data', 'AC_Impedance(Ohm)', 'ACI_Phase_Angle(Deg)']


,Data_Point,Test_Time(s),Date_Time,Step_Time(s),Step_Index,Cycle_Index,Current(A),Voltage(V),Charge_Capacity(Ah),Discharge_Capacity(Ah),Charge_Energy(Wh),Discharge_Energy(Wh),dV/dt(V/s),Internal_Resistance(Ohm),Is_FC_Data,AC_Impedance(Ohm),ACI_Phase_Angle(Deg)
0,1,30.014668,2010-10-22 13:23:55,30.014668,1,1,0.000000,3.669818,0.000000,0,0.000000,0,0.000000,0,0,0,0
1,2,60.029835,2010-10-22 13:24:25,60.029835,1,1,0.000000,3.669818,0.000000,0,0.000000,0,0.000000,0,0,0,0
2,3,90.044914,2010-10-22 13:24:55,90.044914,1,1,0.000000,3.669818,0.000000,0,0.000000,0,0.000032,0,0,0,0
3,4,120.013174,2010-10-22 13:25:25,120.013175,1,1,0.000000,3.669818,0.000000,0,0.000000,0,0.000000,0,0,0,0
4,5,150.028851,2010-10-22 13:25:55,30.015264,2,1,0.675509,3.796711,0.005631,0,0.021304,0,0.000519,0,0,0,0


## 6. Mobile2Mobile Parquet inspection

In [52]:
mobile_files = sorted(MOBILE_SELECTED_DIR.glob("*.parquet"))

print("Selected Parquet files:", len(mobile_files))
print(
    "Selected disk size (MB):",
    round(sum(p.stat().st_size for p in mobile_files) / 1024**2, 2),
)

if not mobile_files:
    raise FileNotFoundError(
        "No selected Mobile2Mobile Parquet files found under "
        f"{MOBILE_SELECTED_DIR}"
    )

first_pf = pq.ParquetFile(mobile_files[0])

print("\nSchema of first file:")
print(first_pf.schema_arrow)

first_batch = next(first_pf.iter_batches(batch_size=10), None)

if first_batch is not None:
    display(first_batch.to_pandas())

parts = []

for path in mobile_files[:10]:
    pf = pq.ParquetFile(path)
    batch = next(pf.iter_batches(batch_size=5_000), None)

    if batch is not None:
        parts.append(batch.to_pandas())

mobile_sample = (
    pd.concat(parts, ignore_index=True)
    if parts
    else pd.DataFrame()
)

if not mobile_sample.empty:
    print("Sample shape:", mobile_sample.shape)

    display(
        (mobile_sample.isna().mean() * 100)
        .sort_values(ascending=False)
        .rename("missing_pct")
    )

    display(mobile_sample.describe(include="all").T)

    for col in ["manufacturer", "model", "battery_technology", "kind"]:
        if col in mobile_sample:
            print(f"\n{col}:")
            display(mobile_sample[col].value_counts(dropna=False).head(20))

Selected Parquet files: 28
Selected disk size (MB): 193.08

Schema of first file:
cell_id: string not null
timestamp: timestamp[us, tz=UTC] not null
battery_technology: string
manufacturer: string
model: string
operating_system: string
os_version: string
android_sdk_version: double
charge_counter: double
nominal_capacity: double
state_of_charge: double
current: double
current_avg: double
plugged: double
temperature_cell: double
voltage_cell: double
cycle_count: double
status: double
health: double
kind: string


,cell_id,timestamp,battery_technology,manufacturer,model,operating_system,os_version,android_sdk_version,charge_counter,nominal_capacity,state_of_charge,current,current_avg,plugged,temperature_cell,voltage_cell,cycle_count,status,health,kind
0,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:55:50.520+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3605,4.9,96.0,-0.937,-0.917,0.0,29.0,4.107,0.0,3.0,2.0,cycling
1,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:56:00.527+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3605,4.9,96.0,-0.272,-0.412,0.0,29.0,4.107,0.0,3.0,2.0,cycling
2,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:56:10.538+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3515,4.9,96.0,-0.352,-0.537,0.0,29.0,4.107,0.0,3.0,2.0,cycling
3,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:56:20.552+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3515,4.9,96.0,-0.160,-0.182,0.0,29.0,4.107,0.0,3.0,2.0,cycling
4,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:56:30.562+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3515,4.9,96.0,-1.237,-1.167,0.0,29.0,4.107,0.0,3.0,2.0,cycling
5,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:56:40.578+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3470,4.9,96.0,-0.377,-0.382,0.0,29.2,4.015,0.0,3.0,2.0,cycling
6,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:56:50.588+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3470,4.9,96.0,-0.545,-0.412,0.0,29.2,4.015,0.0,3.0,2.0,cycling
7,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:57:00.592+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3470,4.9,96.0,-0.545,-0.310,0.0,29.2,4.015,0.0,3.0,2.0,cycling
8,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:57:10.598+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3380,4.9,96.0,-0.607,-0.657,0.0,29.2,4.093,0.0,3.0,2.0,cycling
9,038724c3ec382abe1e57f02f8523a218,2026-05-04 11:57:21.542+00:00,Li-ion,samsung,SM-A146B,NaN,15,35.0,4.3380,4.9,96.0,-0.907,-0.945,0.0,29.2,4.093,0.0,3.0,2.0,cycling


Sample shape: (48750, 20)


operating_system       17.948718
current_avg            10.256410
cell_id                 0.000000
timestamp               0.000000
manufacturer            0.000000
battery_technology      0.000000
os_version              0.000000
model                   0.000000
charge_counter          0.000000
nominal_capacity        0.000000
state_of_charge         0.000000
android_sdk_version     0.000000
current                 0.000000
plugged                 0.000000
temperature_cell        0.000000
voltage_cell            0.000000
cycle_count             0.000000
status                  0.000000
health                  0.000000
kind                    0.000000
Name: missing_pct, dtype: float64

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
cell_id,48750,10,038724c3ec382abe1e57f02f8523a218,5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
timestamp,48750,NaN,NaN,NaN,2026-05-05 09:34:51.387561+00:00,2026-04-29 01:26:06.265000+00:00,2026-04-30 18:21:22.208500+00:00,2026-05-04 08:18:37.081000+00:00,2026-05-06 03:11:45.482000+00:00,2026-09-03 17:53:59.671000+00:00,NaN
battery_technology,48750,1,Li-ion,48750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
manufacturer,48750,1,samsung,48750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,48750,10,SM-A146B,5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
operating_system,40000,8,samsung/a22nsxx/a22:13/TP1A.220624.014/A225FXX...,5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
os_version,48750,4,14,18750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
android_sdk_version,48750.0,NaN,NaN,NaN,34.410256,33.0,34.0,34.0,35.0,36.0,1.030767
charge_counter,48750.0,NaN,NaN,NaN,2.501369,0.04,1.39,2.25,3.49775,5.0,1.349436
nominal_capacity,48750.0,NaN,NaN,NaN,4.880385,4.86,4.86,4.86,4.9,4.905,0.020982



manufacturer:


... (3 outputs left out)


battery_technology:


battery_technology
Li-ion    48750
Name: count, dtype: int64


kind:


kind
cycling    48750
Name: count, dtype: int64

## 7. Unified measurement-level schema

The CSV will have one row per observed time point. Fields that a dataset does not measure remain blank. This is deliberate: laboratory cells and smartphone batteries have different instrumentation.

`measured_capacity_Ah` and `soh_pct` are populated only when a reliable measured capacity can be identified. `capacity_raw_Ah` preserves generic source capacity-like values without claiming they represent a full-discharge capacity.

In [53]:
UNIFIED_COLUMNS = [
    "source_dataset", "domain", "entity_id", "source_file", "record_id",
    "cycle_id", "session_id", "operation", "timestamp_utc", "elapsed_time_s",
    "voltage_V", "current_A", "current_avg_A", "temperature_C", "soc_pct",
    "charge_counter_Ah", "nominal_capacity_Ah", "capacity_raw_Ah",
    "measured_capacity_Ah", "soh_pct", "impedance_re_ohm", "impedance_rct_ohm",
    "manufacturer", "device_model", "battery_technology", "cycle_count",
    "plugged", "status", "health", "kind",
]

# Explicit types keep all source fragments compatible in one Parquet file.
STRING_COLUMNS = [
    "source_dataset", "domain", "entity_id", "source_file", "cycle_id",
    "session_id", "operation", "manufacturer", "device_model",
    "battery_technology", "kind",
]

INTEGER_COLUMNS = ["record_id"]

TIMESTAMP_COLUMNS = ["timestamp_utc"]

FLOAT_COLUMNS = [
    col for col in UNIFIED_COLUMNS
    if col not in STRING_COLUMNS + INTEGER_COLUMNS + TIMESTAMP_COLUMNS
]

ARROW_TYPE_BY_COLUMN = {
    **{col: pa.string() for col in STRING_COLUMNS},
    **{col: pa.int64() for col in INTEGER_COLUMNS},
    **{col: pa.timestamp("ns", tz="UTC") for col in TIMESTAMP_COLUMNS},
    **{col: pa.float64() for col in FLOAT_COLUMNS},
}

ARROW_SCHEMA = pa.schema(
    [pa.field(col, ARROW_TYPE_BY_COLUMN[col]) for col in UNIFIED_COLUMNS]
)

print("Columns:", len(UNIFIED_COLUMNS))
print(UNIFIED_COLUMNS)

print("\nArrow schema:")
print(ARROW_SCHEMA)

NASA_NOMINAL_CAPACITY_AH = 2.0
CALCE_NOMINAL_CAPACITY_AH = {"CS2": 1.10, "CX2": 1.35}

Columns: 30
['source_dataset', 'domain', 'entity_id', 'source_file', 'record_id', 'cycle_id', 'session_id', 'operation', 'timestamp_utc', 'elapsed_time_s', 'voltage_V', 'current_A', 'current_avg_A', 'temperature_C', 'soc_pct', 'charge_counter_Ah', 'nominal_capacity_Ah', 'capacity_raw_Ah', 'measured_capacity_Ah', 'soh_pct', 'impedance_re_ohm', 'impedance_rct_ohm', 'manufacturer', 'device_model', 'battery_technology', 'cycle_count', 'plugged', 'status', 'health', 'kind']

Arrow schema:
source_dataset: string
domain: string
entity_id: string
source_file: string
record_id: int64
cycle_id: string
session_id: string
operation: string
timestamp_utc: timestamp[ns, tz=UTC]
elapsed_time_s: double
voltage_V: double
current_A: double
current_avg_A: double
temperature_C: double
soc_pct: double
charge_counter_Ah: double
nominal_capacity_Ah: double
capacity_raw_Ah: double
measured_capacity_Ah: double
soh_pct: double
impedance_re_ohm: double
impedance_rct_ohm: double
manufacturer: string
device_model:

## 8. Shared conversion helpers

In [54]:
def numeric_array(value):
    if value is None:
        return np.array([], dtype=float)

    try:
        arr = np.asarray(value)

        if arr.dtype == object:
            arr = np.asarray([
                np.nan if v is None else v
                for v in arr.ravel()
            ])

        arr = np.asarray(arr, dtype=float).squeeze()

        return arr.reshape(1) if arr.ndim == 0 else arr.reshape(-1)

    except (TypeError, ValueError):
        return np.array([], dtype=float)


def scalar_number(value):
    arr = numeric_array(value)

    finite = arr[np.isfinite(arr)] if len(arr) else np.array([])

    return float(finite[-1]) if len(finite) else np.nan


def pad_array(arr, n):
    arr = np.asarray(arr).reshape(-1)

    out = np.full(n, np.nan)
    out[:min(n, len(arr))] = arr[:min(n, len(arr))]

    return out


def numeric_series(series, n):
    if series is None:
        return np.full(n, np.nan)

    vals = pd.to_numeric(series, errors="coerce").to_numpy()

    return pad_array(vals, n)


def operation_from_text(value):
    if value is None or (np.isscalar(value) and pd.isna(value)):
        return "UNKNOWN"

    text = str(value).strip().lower()

    if "discharg" in text:
        return "DISCHARGE"

    if "charg" in text:
        return "CHARGE"

    if any(x in text for x in ["rest", "idle", "pause", "ocv"]):
        return "IDLE"

    if "impedance" in text or text == "eis":
        return "IMPEDANCE"

    return "UNKNOWN"


def base_frame(n, source, domain, entity, source_file):
    return pd.DataFrame({
        "source_dataset": [source] * n,
        "domain": [domain] * n,
        "entity_id": [entity] * n,
        "source_file": [source_file] * n,
    })


def normalize_frame(frame):
    """
    Return a frame with exactly the unified columns and expected types.
    """
    frame = frame.reindex(columns=UNIFIED_COLUMNS).copy()

    # Text columns
    for col in STRING_COLUMNS:
        frame[col] = frame[col].astype("string")

    # Nullable integer columns
    for col in INTEGER_COLUMNS:
        frame[col] = pd.to_numeric(
            frame[col],
            errors="coerce"
        ).astype("Int64")

    # UTC timestamps
    for col in TIMESTAMP_COLUMNS:
        frame[col] = pd.to_datetime(
            frame[col],
            errors="coerce",
            utc=True
        )

    # Numeric measurement columns
    for col in FLOAT_COLUMNS:
        values = pd.to_numeric(
            frame[col],
            errors="coerce"
        )

        frame[col] = (
            values
            .replace([np.inf, -np.inf], np.nan)
            .astype("float64")
        )

    return frame


def write_frame(frame, writer, state, row_group_size=50_000):
    if frame is None or frame.empty:
        return 0

    frame = normalize_frame(frame)

    table = pa.Table.from_pandas(
        frame,
        schema=ARROW_SCHEMA,
        preserve_index=False,
        safe=True,
    )

    writer.write_table(
        table,
        row_group_size=row_group_size
    )

    rows_written = len(frame)
    state["rows"] += rows_written

    return rows_written

## 9. NASA loader

In [55]:
def iter_nasa_frames(path):
    mat = load_mat_file(path)
    battery = plain(mat.get(path.stem))
    if not isinstance(battery, dict):
        candidates = [plain(mat[k]) for k in mat if not k.startswith("__")]
        battery = next((x for x in candidates if isinstance(x, dict)), None)
    if not isinstance(battery, dict):
        raise ValueError(f"Could not identify battery structure in {path.name}")

    cycles = to_list(get_key(battery, "cycle"))
    entity = path.stem

    for cycle_idx, cycle in enumerate(cycles, start=1):
        cycle = plain(cycle)
        if not isinstance(cycle, dict):
            continue
        data = plain(get_key(cycle, "data"))
        if not isinstance(data, dict):
            continue

        op = operation_from_text(get_key(cycle, "type"))
        time = numeric_array(get_key(data, "Time"))
        voltage = numeric_array(get_key(data, "Voltage_measured", "Voltage"))
        current = numeric_array(get_key(data, "Current_measured", "Current"))
        temp = numeric_array(get_key(data, "Temperature_measured", "Temperature"))
        n = max(map(len, [time, voltage, current, temp]))
        if n == 0:
            continue

        frame = base_frame(n, "NASA", "LAB", entity, str(path.relative_to(DATA_DIR)))
        frame["record_id"] = np.arange(n)
        frame["cycle_id"] = cycle_idx
        frame["session_id"] = f"{entity}_cycle_{cycle_idx}"
        frame["operation"] = op
        frame["elapsed_time_s"] = pad_array(time, n)
        frame["voltage_V"] = pad_array(voltage, n)
        frame["current_A"] = pad_array(current, n)
        frame["temperature_C"] = pad_array(temp, n)
        frame["nominal_capacity_Ah"] = NASA_NOMINAL_CAPACITY_AH

        cap = scalar_number(get_key(data, "Capacity"))
        if np.isfinite(cap) and op == "DISCHARGE":
            frame["capacity_raw_Ah"] = cap
            frame["measured_capacity_Ah"] = cap
            frame["soh_pct"] = cap / NASA_NOMINAL_CAPACITY_AH * 100

        frame["impedance_re_ohm"] = scalar_number(get_key(data, "Re"))
        frame["impedance_rct_ohm"] = scalar_number(get_key(data, "Rct"))
        yield frame

## 10. CALCE loader

In [56]:
def norm_col(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())

CALCE_ALIASES = {
    "timestamp": ["timestamp", "datetime", "date_time", "date/time", "date"],
    "elapsed": ["test_time(s)", "test time (s)", "test_time", "time(s)", "time_sec", "time_s"],
    "voltage": ["voltage(v)", "voltage (v)", "voltage", "voltage_measured"],
    "current": ["current(a)", "current (a)", "current", "current_measured"],
    "temperature": ["temperature(c)", "temperature (c)", "temperature", "temperature_measured"],
    "cycle": ["cycle_index", "cycle index", "cycle", "cycle_number"],
    "operation": ["operation", "step_type", "step_name", "test_type", "mode"],
    "capacity_raw": ["capacity(ah)", "capacity (ah)", "capacity_ah", "capacity"],
    "measured_capacity": ["discharge_capacity(ah)", "discharge capacity (ah)",
                          "discharge_capacity_ah", "measured_capacity_ah"],
}


def find_column(df, aliases):
    lookup = {norm_col(c): c for c in df.columns}
    for alias in aliases:
        if norm_col(alias) in lookup:
            return lookup[norm_col(alias)]
    return None


def detect_header_row(path, sheet_name=None):
    if path.suffix.lower() in {".xlsx", ".xls"}:
        preview = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=20)
    elif path.suffix.lower() == ".csv":
        preview = pd.read_csv(path, header=None, nrows=20)
    else:
        preview = pd.read_csv(path, sep=None, engine="python", header=None, nrows=20)
    keywords = ["voltage", "current", "temperature", "testtime", "cycleindex", "capacity"]
    scores = []
    for idx, row in preview.iterrows():
        tokens = [norm_col(x) for x in row.tolist() if pd.notna(x)]
        scores.append(sum(any(k in t for k in keywords) for t in tokens))
    return int(np.argmax(scores)) if scores and max(scores) > 0 else 0


def read_calce_table(path, sheet_name=None):
    header = detect_header_row(path, sheet_name)
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, sheet_name=sheet_name, header=header)
    elif suffix == ".csv":
        df = pd.read_csv(path, header=header)
    else:
        df = pd.read_csv(path, sep=None, engine="python", header=header)
    df = df.dropna(how="all")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def iter_calce_frames(path):
    rel = path.relative_to(CALCE_DIR)
    entity = rel.parts[0] if len(rel.parts) > 1 else path.stem.split("_")[0]
    nominal = CALCE_NOMINAL_CAPACITY_AH.get(entity.upper()[:3], np.nan)

    if path.suffix.lower() in {".xlsx", ".xls"}:
        try:
            sheets = pd.ExcelFile(path).sheet_names
        except Exception as exc:
            print(f"Workbook error {path}: {exc}")
            return
    else:
        sheets = [None]

    for sheet in sheets:
        try:
            df = read_calce_table(path, sheet)
        except Exception as exc:
            print(f"Read error {path.name}/{sheet}: {exc}")
            continue
        if df.empty:
            continue

        cols = {k: find_column(df, aliases) for k, aliases in CALCE_ALIASES.items()}
        if not any(cols[k] for k in ["voltage", "current", "temperature", "elapsed", "timestamp"]):
            continue

        n = len(df)
        frame = base_frame(n, "CALCE", "LAB", entity, str(path.relative_to(DATA_DIR)))
        frame["record_id"] = np.arange(n)
        frame["session_id"] = path.stem + (f"_{sheet}" if sheet is not None else "")
        frame["nominal_capacity_Ah"] = nominal

        for target, key in [("voltage_V", "voltage"), ("current_A", "current"),
                            ("temperature_C", "temperature"), ("elapsed_time_s", "elapsed")]:
            col = cols[key]
            frame[target] = numeric_series(df[col] if col else None, n)

        if cols["cycle"]:
            frame["cycle_id"] = df[cols["cycle"]].to_numpy()
        if cols["operation"]:
            frame["operation"] = df[cols["operation"]].map(operation_from_text)
        else:
            frame["operation"] = "UNKNOWN"
        if cols["capacity_raw"]:
            frame["capacity_raw_Ah"] = numeric_series(df[cols["capacity_raw"]], n)
        if cols["measured_capacity"]:
            measured = numeric_series(df[cols["measured_capacity"]], n)
            frame["measured_capacity_Ah"] = measured
            if np.isfinite(nominal) and nominal > 0:
                frame["soh_pct"] = measured / nominal * 100

        if cols["timestamp"]:
            raw_time = df[cols["timestamp"]]
            if pd.api.types.is_numeric_dtype(raw_time):
                # Excel serial date representation (if this is indeed a date column).
                frame["timestamp_utc"] = pd.to_datetime(
                    raw_time, unit="D", origin="1899-12-30", errors="coerce", utc=True
                )
            else:
                frame["timestamp_utc"] = pd.to_datetime(raw_time, errors="coerce", utc=True)
        yield frame

## 11. Mobile2Mobile loader

In [57]:
def iter_mobile_frames(path, batch_size=100_000):
    """
    Stream one Mobile2Mobile Parquet file in batches.

    record_id increases across batches within this source file, rather than
    restarting at zero for every batch.
    """
    pf = pq.ParquetFile(path)

    # Find a timestamp baseline from a small initial sample.
    first_batch = next(
        pf.iter_batches(
            batch_size=10_000,
            columns=["timestamp"]
        ),
        None,
    )

    first_ts = None

    if first_batch is not None and first_batch.num_rows:
        sample_ts = pd.to_datetime(
            pd.Series(first_batch.column(0).to_pylist()),
            errors="coerce",
            utc=True,
        ).dropna()

        if not sample_ts.empty:
            first_ts = sample_ts.min()

    record_offset = 0

    for batch in pf.iter_batches(batch_size=batch_size):
        df = batch.to_pandas()
        n = len(df)

        if n == 0:
            continue

        fallback_entity = path.stem

        frame = base_frame(
            n,
            "MOBILE2MOBILE",
            "REAL_WORLD",
            fallback_entity,
            str(path.relative_to(DATA_DIR)),
        )

        # Keep entity ID row-specific in case a file contains >1 cell_id.
        if "cell_id" in df:
            frame["entity_id"] = (
                df["cell_id"]
                .astype("string")
                .fillna(fallback_entity)
                .to_numpy()
            )

        frame["record_id"] = np.arange(
            record_offset,
            record_offset + n,
            dtype=np.int64,
        )

        record_offset += n

        if "timestamp" in df:
            ts = pd.to_datetime(
                df["timestamp"],
                errors="coerce",
                utc=True
            )

            frame["timestamp_utc"] = ts

            if first_ts is not None:
                frame["elapsed_time_s"] = (
                    ts - first_ts
                ).dt.total_seconds()

        mapping = {
            "voltage_V": "voltage_cell",
            "current_A": "current",
            "current_avg_A": "current_avg",
            "temperature_C": "temperature_cell",
            "soc_pct": "state_of_charge",
            "charge_counter_Ah": "charge_counter",
            "nominal_capacity_Ah": "nominal_capacity",
            "cycle_count": "cycle_count",
            "plugged": "plugged",
            "status": "status",
            "health": "health",
            "kind": "kind",
            "manufacturer": "manufacturer",
            "device_model": "model",
            "battery_technology": "battery_technology",
        }

        for target, source in mapping.items():
            if source in df:
                frame[target] = df[source].to_numpy()

        if "kind" in df:
            frame["operation"] = df["kind"].map(operation_from_text)
        yield frame

## 12. Preview one fragment from each loader

In [58]:
def preview_generator(generator, label):
    try:
        frame = next(iter(generator))
        print(f"{label}: {frame.shape}")
        display(frame.head(5))
        display(frame.notna().sum().rename("non_null_count").to_frame())
    except StopIteration:
        print(f"{label}: no rows emitted")
    except Exception as exc:
        print(f"{label} preview failed: {exc!r}")

preview_generator(iter_nasa_frames(nasa_files[0]), "NASA")
calce_preview_path = next((p for p in calce_files if p.suffix.lower() in {".xlsx", ".xls"}), None)
if calce_preview_path:
    preview_generator(iter_calce_frames(calce_preview_path), "CALCE")
preview_generator(iter_mobile_frames(mobile_files[0]), "MOBILE2MOBILE")

NASA: (789, 15)


,source_dataset,domain,entity_id,source_file,record_id,cycle_id,session_id,operation,elapsed_time_s,voltage_V,current_A,temperature_C,nominal_capacity_Ah,impedance_re_ohm,impedance_rct_ohm
0,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,0,1,B0005_cycle_1,CHARGE,0.000,3.873017,-0.001201,24.655358,2.0,NaN,NaN
1,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,1,1,B0005_cycle_1,CHARGE,2.532,3.479394,-4.030268,24.666480,2.0,NaN,NaN
2,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,2,1,B0005_cycle_1,CHARGE,5.500,4.000588,1.512731,24.675394,2.0,NaN,NaN
3,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,3,1,B0005_cycle_1,CHARGE,8.344,4.012395,1.509063,24.693865,2.0,NaN,NaN
4,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,4,1,B0005_cycle_1,CHARGE,11.125,4.019708,1.511318,24.705069,2.0,NaN,NaN


,non_null_count
source_dataset,789
domain,789
entity_id,789
source_file,789
record_id,789
cycle_id,789
session_id,789
operation,789
elapsed_time_s,789
voltage_V,789


CALCE: (22908, 16)


,source_dataset,domain,entity_id,source_file,record_id,session_id,nominal_capacity_Ah,voltage_V,current_A,temperature_C,elapsed_time_s,cycle_id,operation,measured_capacity_Ah,soh_pct,timestamp_utc
0,CALCE,LAB,CS2_33,calce/CS2_33/CS2_33/CS2_33_10_26_10.xlsx,0,CS2_33_10_26_10_Channel_1-006,1.1,3.520348,0.000000,NaN,30.015247,1,UNKNOWN,0.0,0.0,2010-10-15 17:12:56+00:00
1,CALCE,LAB,CS2_33,calce/CS2_33/CS2_33/CS2_33_10_26_10.xlsx,1,CS2_33_10_26_10_Channel_1-006,1.1,3.520348,0.000000,NaN,60.030234,1,UNKNOWN,0.0,0.0,2010-10-15 17:13:26+00:00
2,CALCE,LAB,CS2_33,calce/CS2_33/CS2_33/CS2_33_10_26_10.xlsx,2,CS2_33_10_26_10_Channel_1-006,1.1,3.520348,0.000000,NaN,90.045398,1,UNKNOWN,0.0,0.0,2010-10-15 17:13:56+00:00
3,CALCE,LAB,CS2_33,calce/CS2_33/CS2_33/CS2_33_10_26_10.xlsx,3,CS2_33_10_26_10_Channel_1-006,1.1,3.520510,0.000000,NaN,120.013513,1,UNKNOWN,0.0,0.0,2010-10-15 17:14:26+00:00
4,CALCE,LAB,CS2_33,calce/CS2_33/CS2_33/CS2_33_10_26_10.xlsx,4,CS2_33_10_26_10_Channel_1-006,1.1,3.660633,0.550205,NaN,150.029139,1,UNKNOWN,0.0,0.0,2010-10-15 17:14:56+00:00


,non_null_count
source_dataset,22908
domain,22908
entity_id,22908
source_file,22908
record_id,22908
session_id,22908
nominal_capacity_Ah,22908
voltage_V,22908
current_A,22908
temperature_C,0


MOBILE2MOBILE: (100000, 23)


,source_dataset,domain,entity_id,source_file,record_id,timestamp_utc,elapsed_time_s,voltage_V,current_A,current_avg_A,...,nominal_capacity_Ah,cycle_count,plugged,status,health,kind,manufacturer,device_model,battery_technology,operation
0,MOBILE2MOBILE,REAL_WORLD,038724c3ec382abe1e57f02f8523a218,mobile2mobile/selected_data/038724c3ec382abe1e...,0,2026-05-04 11:55:50.520+00:00,0.000,4.107,-0.937,-0.917,...,4.9,0.0,0.0,3.0,2.0,cycling,samsung,SM-A146B,Li-ion,UNKNOWN
1,MOBILE2MOBILE,REAL_WORLD,038724c3ec382abe1e57f02f8523a218,mobile2mobile/selected_data/038724c3ec382abe1e...,1,2026-05-04 11:56:00.527+00:00,10.007,4.107,-0.272,-0.412,...,4.9,0.0,0.0,3.0,2.0,cycling,samsung,SM-A146B,Li-ion,UNKNOWN
2,MOBILE2MOBILE,REAL_WORLD,038724c3ec382abe1e57f02f8523a218,mobile2mobile/selected_data/038724c3ec382abe1e...,2,2026-05-04 11:56:10.538+00:00,20.018,4.107,-0.352,-0.537,...,4.9,0.0,0.0,3.0,2.0,cycling,samsung,SM-A146B,Li-ion,UNKNOWN
3,MOBILE2MOBILE,REAL_WORLD,038724c3ec382abe1e57f02f8523a218,mobile2mobile/selected_data/038724c3ec382abe1e...,3,2026-05-04 11:56:20.552+00:00,30.032,4.107,-0.160,-0.182,...,4.9,0.0,0.0,3.0,2.0,cycling,samsung,SM-A146B,Li-ion,UNKNOWN
4,MOBILE2MOBILE,REAL_WORLD,038724c3ec382abe1e57f02f8523a218,mobile2mobile/selected_data/038724c3ec382abe1e...,4,2026-05-04 11:56:30.562+00:00,40.042,4.107,-1.237,-1.167,...,4.9,0.0,0.0,3.0,2.0,cycling,samsung,SM-A146B,Li-ion,UNKNOWN


,non_null_count
source_dataset,100000
domain,100000
entity_id,100000
source_file,100000
record_id,100000
timestamp_utc,100000
elapsed_time_s,100000
voltage_V,100000
current_A,100000
current_avg_A,100000


## 13. Export all sources to unified.parquet

In [59]:
OVERWRITE_UNIFIED_PARQUET = True

if OUTPUT_PARQUET.exists() and not OVERWRITE_UNIFIED_PARQUET:
    raise FileExistsError(
        f"{OUTPUT_PARQUET} already exists. "
        "Set OVERWRITE_UNIFIED_PARQUET=True to replace it."
    )

# Write to a temporary file first so an interrupted run does not overwrite
# a previously completed unified.parquet file.
TEMP_PARQUET = OUTPUT_PARQUET.with_name(
    OUTPUT_PARQUET.stem + ".inprogress" + OUTPUT_PARQUET.suffix
)

if TEMP_PARQUET.exists():
    TEMP_PARQUET.unlink()

state = {"rows": 0}
source_counts = Counter()
errors = []

writer = pq.ParquetWriter(
    TEMP_PARQUET,
    schema=ARROW_SCHEMA,
    compression="zstd",
    compression_level=6,
    use_dictionary=STRING_COLUMNS,
    write_statistics=True,
    version="2.6",
)

try:
    print("Processing NASA...")

    for i, path in enumerate(nasa_files, 1):
        try:
            for frame in iter_nasa_frames(path):
                source_counts["NASA"] += write_frame(
                    frame,
                    writer,
                    state
                )

        except Exception as exc:
            errors.append(("NASA", str(path), repr(exc)))
            print(f"NASA ERROR {path.name}: {exc}")

        if i % 10 == 0 or i == len(nasa_files):
            print(f"  {i}/{len(nasa_files)} files")

    print("\nProcessing CALCE...")

    for i, path in enumerate(calce_files, 1):
        try:
            for frame in iter_calce_frames(path):
                source_counts["CALCE"] += write_frame(
                    frame,
                    writer,
                    state
                )

        except Exception as exc:
            errors.append(("CALCE", str(path), repr(exc)))
            print(f"CALCE ERROR {path.name}: {exc}")

        if i % 25 == 0 or i == len(calce_files):
            print(f"  {i}/{len(calce_files)} files")

    print("\nProcessing Mobile2Mobile...")

    for i, path in enumerate(mobile_files, 1):
        try:
            for frame in iter_mobile_frames(path):
                source_counts["MOBILE2MOBILE"] += write_frame(
                    frame,
                    writer,
                    state
                )

        except Exception as exc:
            errors.append(("MOBILE2MOBILE", str(path), repr(exc)))
            print(f"MOBILE ERROR {path.name}: {exc}")

        if i % 10 == 0 or i == len(mobile_files):
            print(f"  {i}/{len(mobile_files)} files")

finally:
    writer.close()

TEMP_PARQUET.replace(OUTPUT_PARQUET)

print("\n========== EXPORT SUMMARY ==========")
print("Output:", OUTPUT_PARQUET)
print("Rows:", f"{state['rows']:,}")
print(
    "Size (MB):",
    round(OUTPUT_PARQUET.stat().st_size / 1024**2, 2)
)
print("Rows by source:", dict(source_counts))
print("Errors:", len(errors))

if errors:
    error_df = pd.DataFrame(
        errors,
        columns=["source", "file", "error"]
    )

    display(error_df)

    error_path = DATA_DIR / "unified_export_errors.csv"
    error_df.to_csv(error_path, index=False)

    print("Error log:", error_path)

else:
    print("No source files failed.")

Processing NASA...
  10/38 files
  20/38 files
  30/38 files
  38/38 files

Processing CALCE...
  25/88 files
  50/88 files
  75/88 files
  88/88 files

Processing Mobile2Mobile...
  10/28 files
  20/28 files
  28/28 files

========== EXPORT SUMMARY ==========
Output: /home/raj/Documents/Projects/PBL/data/unified.parquet
Rows: 26,946,325
Size (MB): 508.1
Rows by source: {'NASA': 7699426, 'CALCE': 1208143, 'MOBILE2MOBILE': 18038756}
Errors: 0
No source files failed.


## 14. Validate the generated Parquet

In [60]:
if not OUTPUT_PARQUET.exists():
    raise FileNotFoundError(
        f"{OUTPUT_PARQUET} does not exist. Run the export cell first."
    )

parquet_file = pq.ParquetFile(OUTPUT_PARQUET)
metadata = parquet_file.metadata

print("Output:", OUTPUT_PARQUET)
print("Rows:", f"{metadata.num_rows:,}")
print("Row groups:", metadata.num_row_groups)
print("Columns:", parquet_file.schema_arrow.names)
print(
    "Size (MB):",
    round(OUTPUT_PARQUET.stat().st_size / 1024**2, 2)
)

print("\nArrow schema:")
print(parquet_file.schema_arrow)

# Preview only the first 10 records.
preview_batch = next(
    parquet_file.iter_batches(batch_size=10),
    None
)

if preview_batch is not None:
    preview = preview_batch.to_pandas()
    display(preview)

source_rows = Counter()
labelled_rows = Counter()
entity_sets = defaultdict(set)

validation_columns = [
    "source_dataset",
    "entity_id",
    "soh_pct",
]

for batch in parquet_file.iter_batches(
    batch_size=250_000,
    columns=validation_columns,
):
    sample = batch.to_pandas()

    for source, group in sample.groupby(
        "source_dataset",
        dropna=False
    ):
        source_key = (
            "(missing)"
            if pd.isna(source)
            else str(source)
        )

        source_rows[source_key] += len(group)

        labelled_rows[source_key] += int(
            group["soh_pct"].notna().sum()
        )

        entity_sets[source_key].update(
            group["entity_id"]
            .dropna()
            .astype(str)
            .unique()
        )

summary = pd.DataFrame([
    {
        "source_dataset": source,
        "rows": source_rows[source],
        "rows_with_soh_label": labelled_rows[source],
        "distinct_entities": len(entity_sets[source]),
        "soh_label_pct": (
            100 * labelled_rows[source] / source_rows[source]
            if source_rows[source]
            else 0.0
        ),
    }
    for source in sorted(source_rows)
])

print("\nSource-level validation:")
display(summary)

print(
    "\nTotal rows checked:",
    f"{sum(source_rows.values()):,}"
)

print(
    "Total rows with SOH label:",
    f"{sum(labelled_rows.values()):,}"
)

Output: /home/raj/Documents/Projects/PBL/data/unified.parquet
Rows: 26,946,325
Row groups: 6363
Columns: ['source_dataset', 'domain', 'entity_id', 'source_file', 'record_id', 'cycle_id', 'session_id', 'operation', 'timestamp_utc', 'elapsed_time_s', 'voltage_V', 'current_A', 'current_avg_A', 'temperature_C', 'soc_pct', 'charge_counter_Ah', 'nominal_capacity_Ah', 'capacity_raw_Ah', 'measured_capacity_Ah', 'soh_pct', 'impedance_re_ohm', 'impedance_rct_ohm', 'manufacturer', 'device_model', 'battery_technology', 'cycle_count', 'plugged', 'status', 'health', 'kind']
Size (MB): 508.1

Arrow schema:
source_dataset: string
domain: string
entity_id: string
source_file: string
record_id: int64
cycle_id: string
session_id: string
operation: string
timestamp_utc: timestamp[ns, tz=UTC]
elapsed_time_s: double
voltage_V: double
current_A: double
current_avg_A: double
temperature_C: double
soc_pct: double
charge_counter_Ah: double
nominal_capacity_Ah: double
capacity_raw_Ah: double
measured_capacity_Ah

,source_dataset,domain,entity_id,source_file,record_id,cycle_id,session_id,operation,timestamp_utc,elapsed_time_s,...,impedance_re_ohm,impedance_rct_ohm,manufacturer,device_model,battery_technology,cycle_count,plugged,status,health,kind
0,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,0,1,B0005_cycle_1,CHARGE,NaT,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,1,1,B0005_cycle_1,CHARGE,NaT,2.532,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,2,1,B0005_cycle_1,CHARGE,NaT,5.500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,3,1,B0005_cycle_1,CHARGE,NaT,8.344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,4,1,B0005_cycle_1,CHARGE,NaT,11.125,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,5,1,B0005_cycle_1,CHARGE,NaT,13.891,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,6,1,B0005_cycle_1,CHARGE,NaT,16.672,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,7,1,B0005_cycle_1,CHARGE,NaT,19.500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,8,1,B0005_cycle_1,CHARGE,NaT,22.282,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NASA,LAB,B0005,nasa/extracted/1. BatteryAgingARC-FY08Q4/B0005...,9,1,B0005_cycle_1,CHARGE,NaT,25.063,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Source-level validation:


,source_dataset,rows,rows_with_soh_label,distinct_entities,soh_label_pct
0,CALCE,1208143,1180951,13,97.749273
1,MOBILE2MOBILE,18038756,0,28,0.000000
2,NASA,7699426,830026,34,10.780362



Total rows checked: 26,946,325
Total rows with SOH label: 2,010,977
